In [9]:
import math
import numpy as np
import pandas as pd
from typing import Optional, Dict, Tuple, List

In [10]:
file_path = "/content/data_hist.xlsx"

hist = pd.read_excel(file_path, sheet_name="historical_2023_2025")
forecast_inputs = pd.read_excel(file_path, sheet_name="forecast_2026")
forecast_uq = pd.read_excel("/content/forecast_u_q_2026.xlsx")

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [12]:
forecast_inputs["date"] = forecast_inputs["date"].astype(str)
forecast_uq["date"] = forecast_uq["date"].astype(str)
hist["date"] = hist["date"].astype(str)

forecast_inputs

,date,year,month,vacancy,has_internship,staff_plan,initial_staff_fact_2025_12,initial_interns_2025_12,planned_start_groups,group_limit,internship_capacity,internship_duration_months,contact_lag_days,max_hire_monthly,K_total_expected,K_stage1_expected
0,2026-01,2026,1,Инспектор контроля,True,577,556.0,28.0,2,24,70,3,28,NaN,0.105,0.42
1,2026-02,2026,2,Инспектор контроля,True,577,NaN,NaN,2,24,70,3,28,NaN,0.105,0.42
2,2026-03,2026,3,Инспектор контроля,True,577,NaN,NaN,2,24,70,3,28,NaN,0.105,0.42
3,2026-04,2026,4,Инспектор контроля,True,577,NaN,NaN,2,24,70,3,28,NaN,0.105,0.42
4,2026-05,2026,5,Инспектор контроля,True,577,NaN,NaN,2,24,70,3,28,NaN,0.105,0.42
5,2026-06,2026,6,Инспектор контроля,True,577,NaN,NaN,2,24,70,3,28,NaN,0.105,0.42
6,2026-07,2026,7,Инспектор контроля,True,577,NaN,NaN,2,24,70,3,28,NaN,0.105,0.42
7,2026-08,2026,8,Инспектор контроля,True,578,NaN,NaN,2,24,70,3,28,NaN,0.105,0.42
8,2026-09,2026,9,Инспектор контроля,True,578,NaN,NaN,1,24,70,3,28,NaN,0.105,0.42
9,2026-10,2026,10,Инспектор контроля,True,578,NaN,NaN,1,24,70,3,28,NaN,0.105,0.42


In [16]:
forecast_df = forecast_inputs.merge(forecast_uq [["date", "vacancy", "U_pred", "Q_pred"]], on=["date", "vacancy"], how="left")
forecast_df["Q_pred"] = forecast_df["Q_pred"].fillna(0)
forecast_df.shape

(60, 18)

In [14]:
# входные данные, если запускать в разрезе января 2026
start_state = (hist[hist["date"] == "2025-12"][["vacancy", "staff_fact", "interns_end"]].copy())
start_state = start_state.rename(columns={"staff_fact": "initial_staff", "interns_end": "initial_interns"})

start_state

,vacancy,initial_staff,initial_interns
35,Инспектор контроля,556,28
71,Инспектор безопасности,236,21
107,Оператор техники,101,0
143,Специалист обслуживания,146,0
179,Оператор багажной системы,125,0


In [17]:
forecast_df = forecast_df.merge( start_state, on="vacancy", how="left")
forecast_df["initial_interns"] = forecast_df["initial_interns"].fillna(0)
forecast_df.head()

,date,year,month,vacancy,has_internship,staff_plan,initial_staff_fact_2025_12,initial_interns_2025_12,planned_start_groups,group_limit,internship_capacity,internship_duration_months,contact_lag_days,max_hire_monthly,K_total_expected,K_stage1_expected,U_pred,Q_pred,initial_staff,initial_interns
0,2026-01,2026,1,Инспектор контроля,True,577,556.0,28.0,2,24,70,3,28,NaN,0.105,0.42,0.017700,0.111233,556,28
1,2026-02,2026,2,Инспектор контроля,True,577,NaN,NaN,2,24,70,3,28,NaN,0.105,0.42,0.017865,0.111233,556,28
2,2026-03,2026,3,Инспектор контроля,True,577,NaN,NaN,2,24,70,3,28,NaN,0.105,0.42,0.020780,0.111173,556,28
3,2026-04,2026,4,Инспектор контроля,True,577,NaN,NaN,2,24,70,3,28,NaN,0.105,0.42,0.022813,0.112912,556,28
4,2026-05,2026,5,Инспектор контроля,True,577,NaN,NaN,2,24,70,3,28,NaN,0.105,0.42,0.024014,0.116925,556,28


In [18]:
forecast_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   date                        60 non-null     object 
 1   year                        60 non-null     int64  
 2   month                       60 non-null     int64  
 3   vacancy                     60 non-null     object 
 4   has_internship              60 non-null     bool   
 5   staff_plan                  60 non-null     int64  
 6   initial_staff_fact_2025_12  5 non-null      float64
 7   initial_interns_2025_12     2 non-null      float64
 8   planned_start_groups        60 non-null     int64  
 9   group_limit                 60 non-null     int64  
 10  internship_capacity         60 non-null     int64  
 11  internship_duration_months  60 non-null     int64  
 12  contact_lag_days            60 non-null     int64  
 13  max_hire_monthly            36 non-nu

Сама модель расчета параметров

In [20]:
def plan_staff_model(
    df: pd.DataFrame,
    staff_prev_month: int,

    # Есть ли стажерский период
    has_internship: bool = True,

    # Параметры для вакансий со стажерским периодом
    internship_duration: int = 3,
    group_limit: int = 24,
    internship_capacity: int = 45,
    initial_interns: int = 0,
    scheduled_intern_transfers: Optional[Dict[str, int]] = None,
    employees_in_processing: int = 0,
    contact_wait_days: int = 24,

    # Параметры для вакансий без стажерского периода
    max_hire_no_internship: Optional[int] = None,
    accepted_prev_month_no_internship: int = 0,
    accepted_current_month_no_internship: int = 0,

    # Период расчета
    from_month: Optional[str] = None,
    show_until: Optional[str] = "2026-12",

    # Сценарная корректировка ML-прогноза
    u_by_month: Optional[Dict[str, float]] = None,
    q_by_month: Optional[Dict[str, float]] = None,) -> pd.DataFrame:


    df2 = df.copy()

    if scheduled_intern_transfers is None:
        scheduled_intern_transfers = {}

    if u_by_month is None:
        u_by_month = {}

    if q_by_month is None:
        q_by_month = {}


    # Фильтр периода с заданного параметра
    if from_month is not None:
        month_key = pd.PeriodIndex(pd.to_datetime(df2["date"]), freq="M").astype(str)
        target = str(pd.Period(from_month, freq="M"))

        idx = np.where(month_key.to_numpy() == target)[0]
        df2 = df2.iloc[int(idx[0]):].reset_index(drop=True)

    df2 = df2.sort_values("date").reset_index(drop=True)
    n = len(df2)


    months = df2["date"].astype(str).to_numpy()
    staff_plan = df2["staff_plan"].to_numpy()

    u_col = df2["U_pred"].astype(float).to_numpy()

    if has_internship:
        q_col = df2["Q_pred"].fillna(0).astype(float).to_numpy()
        groups_col = df2["planned_start_groups"].fillna(0).astype(int).to_numpy()
    else:
        q_col = np.zeros(n, dtype=float)
        groups_col = np.zeros(n, dtype=int)

    # Конверсии воронки
    ksum_col = df2["K_total_expected"].astype(float).to_numpy()
    k_stage1_col = df2["K_stage1_expected"].astype(float).to_numpy()

    # Лаг от контакта до выхода/оформления
    contact_lag_months = math.ceil(contact_wait_days / 30) if contact_wait_days > 0 else 0
    backlog = 0.0

    # Массивы результатов
    staff_by_month = np.zeros(n, dtype=int)
    deficit_arr = np.zeros(n, dtype=int)

    hire_arr = np.zeros(n, dtype=int)
    contacts_need = np.zeros(n, dtype=float)
    candidates_need = np.zeros(n, dtype=float)
    hire_out_arr = np.zeros(n, dtype=int)

    # Для вакансий со стажерским периодом
    interns_start_arr = np.zeros(n, dtype=int)
    interns_to_staff_arr = np.zeros(n, dtype=int)
    interns_end_arr = np.zeros(n, dtype=int)

    # Для вакансий без стажерского периода
    pending_start_arr = np.zeros(n, dtype=int)
    pending_out_arr = np.zeros(n, dtype=int)
    pending_end_arr = np.zeros(n, dtype=int)

    # Дополнительная функция выбора коэффициента
    def pick_coef(t: int, base_array: np.ndarray, by_month: Dict[str, float]) -> float:

        m = months[t]

        if m in by_month:
            return float(by_month[m])

        return float(base_array[t])

    #  Стартовые значения
    staff = int(staff_prev_month)
    # стажеры, которые уже были на начало прогноза
    existing_interns = int(initial_interns)
    # стажеры, которых модель нанимает уже внутри прогноза
    interns_queue: List[Tuple[int, int]] = []
    # очередь найма для вакансий без стажерского периода
    pending_hires: List[Tuple[int, int]] = []

    if not has_internship:
        if accepted_prev_month_no_internship > 0:
            pending_hires.append((1, int(accepted_prev_month_no_internship)))

        if accepted_current_month_no_internship > 0:
            pending_hires.append((0, int(accepted_current_month_no_internship)))


    #  Защита от перебора по плану для стажерского периода
    def max_safe_hire_internship(
        hire_cap: int,
        staff_now: int,
        existing_interns_now: int,
        interns_queue_now: List[Tuple[int, int]],
        t_now: int,
        h: int,) -> int:


        def violates(hire0: int) -> bool:
            s = int(staff_now)
            ex = int(existing_interns_now)
            queue = list(interns_queue_now)

            if hire0 > 0:
                queue.append((0, hire0))

            for step in range(1, h + 1):
                i = t_now + step
                ui = pick_coef(i, u_col, u_by_month)
                qi = pick_coef(i, q_col, q_by_month)
                mi = months[i]

                # Стартовые стажеры
                ex_drop = int(round(ex * qi))
                ex_after = ex - ex_drop

                ex_transfer = min(
                    ex_after,
                    int(scheduled_intern_transfers.get(mi, 0)))

                ex = ex_after - ex_transfer
                transfers_i = ex_transfer

                # Стажеры, набранные внутри модели
                next_queue = []

                for age, size in queue:
                    dropped = int(round(size * qi))
                    left = size - dropped
                    age_next = age + 1

                    if age_next >= internship_duration:
                        transfers_i += left
                    else:
                        next_queue.append((age_next, left))

                queue = next_queue

                # Обновление основного штата
                s = int(round(s - s * ui + transfers_i))

                if s > staff_plan[i]:
                    return True

            return False

        if violates(0):
            return 0

        lo, hi = 0, int(hire_cap)

        while lo < hi:
            mid = (lo + hi + 1) // 2

            if violates(mid):
                hi = mid - 1
            else:
                lo = mid

        return lo

    #   цикл по месяцам для сотрудников
    for t in range(n):
        m = months[t]
        plan_t = float(staff_plan[t])
        u = pick_coef(t, u_col, u_by_month)

        if has_internship:
            q = pick_coef(t, q_col, q_by_month)
            groups = int(groups_col[t])

            # Сотрудники в оформлении добавляются в стажерский контур в первом месяце
            processing_now = int(employees_in_processing) if t == 0 else 0
            if processing_now > 0:
                interns_queue.append((0, processing_now))

            interns_start = existing_interns + sum(size for _, size in interns_queue)
            interns_start_arr[t] = interns_start
            existing_drop = int(round(existing_interns * q))
            existing_after = existing_interns - existing_drop
            existing_transfer = min(existing_after, int(scheduled_intern_transfers.get(m, 0)))
            existing_interns = existing_after - existing_transfer
            drops = existing_drop
            transfers = existing_transfer


            next_queue: List[Tuple[int, int]] = []
            for age, size in interns_queue:
                dropped = int(round(size * q))
                left = size - dropped
                drops += dropped
                age_next = age + 1

                if age_next >= internship_duration:
                    transfers += left
                else:
                    next_queue.append((age_next, left))
            interns_queue = next_queue



            staff = int(round(staff - staff * u + transfers))
            staff_by_month[t] = staff
            interns_to_staff_arr[t] = transfers



            deficit_now = max(0.0, plan_t - staff)
            deficit_arr[t] = int(round(deficit_now))
            backlog = max(0.0, backlog + deficit_now)


            h = (n - 1) - t
            need_now = 0.0

            if groups > 0 and h > 0:
                s = int(staff)
                ex = int(existing_interns)
                queue = list(interns_queue)

                for step in range(1, h + 1):
                    i = t + step
                    ui = pick_coef(i, u_col, u_by_month)
                    qi = pick_coef(i, q_col, q_by_month)
                    mi = months[i]
                    ex_drop = int(round(ex * qi))
                    ex_after = ex - ex_drop
                    ex_transfer = min(ex_after, int(scheduled_intern_transfers.get(mi, 0)))
                    ex = ex_after - ex_transfer
                    transfers_i = ex_transfer
                    next_queue_sim = []

                    for age, size in queue:
                        dropped = int(round(size * qi))
                        left = size - dropped
                        age_next = age + 1
                        if age_next >= internship_duration:
                            transfers_i += left
                        else:
                            next_queue_sim.append((age_next, left))

                    queue = next_queue_sim
                    s = int(round(s - s * ui + transfers_i))
                plan_target = float(staff_plan[t + h])
                future_deficit = max(0.0, plan_target - s)

                backlog_limited = min(backlog, 0.2 * internship_capacity)

                need_now = ((future_deficit + backlog_limited) / (1 - q)  if (1 - q) > 0 else 0.0)

            # Ограничения найма стажеров
            out_total = drops + transfers
            free_capacity = internship_capacity - interns_start + out_total

            hire_target = int(min(need_now, groups * group_limit, free_capacity))
            hire_target = int(max(0, hire_target - processing_now))
            hire = hire_target

            if h > 0 and hire > 0:
                hire = max_safe_hire_internship(
                    hire_cap=hire,
                    staff_now=staff,
                    existing_interns_now=existing_interns,
                    interns_queue_now=interns_queue,
                    t_now=t,
                    h=h, )

            hire = max(0, hire)
            hire_arr[t] = hire
            interns_end_arr[t] = interns_start - out_total + hire
            if hire > 0:
                interns_queue.append((0, hire))



        else:
            pending_start_arr[t] = sum(size for _, size in pending_hires)
            hired_to_staff = 0
            next_pending: List[Tuple[int, int]] = []

            for age, size in pending_hires:
                age_next = age + 1
                if age_next >= contact_lag_months:
                    hired_to_staff += size
                else:
                    next_pending.append((age_next, size))
            pending_hires = next_pending

            staff = int(round(staff - staff * u + hired_to_staff))
            staff_by_month[t] = staff
            hire_out_arr[t] = hired_to_staff
            pending_out_arr[t] = hired_to_staff


            need_now = 0.0
            target_idx = t + contact_lag_months
            if target_idx < n:
                target_plan = float(staff_plan[target_idx])

                s = int(staff)
                queue_sim = list(pending_hires)

                for step in range(1, contact_lag_months + 1):
                    i = t + step
                    ui = pick_coef(i, u_col, u_by_month)
                    hired_i = 0
                    queue_next = []
                    for age, size in queue_sim:
                        age_next = age + 1
                        if age_next >= contact_lag_months:
                            hired_i += size
                        else:
                            queue_next.append((age_next, size))
                    queue_sim = queue_next
                    s = int(round(s - s * ui + hired_i))
                future_deficit = max(0.0, target_plan - s)
                need_now = future_deficit


            hire = int(max(0, round(need_now)))
            if target_idx < n:
                hire = min(hire, max(0, int(target_plan - s)))
            if max_hire_no_internship is not None:
                hire = min(hire, int(max_hire_no_internship))
            hire = max(0, hire)
            hire_arr[t] = hire


            if hire > 0:
                if contact_lag_months == 0:
                    staff = int(staff + hire)
                    staff_by_month[t] = staff
                    hire_out_arr[t] += hire
                    pending_out_arr[t] += hire
                else:
                    pending_hires.append((0, hire))

            pending_end_arr[t] = sum(size for _, size in pending_hires)
            deficit_arr[t] = int(max(0, round(plan_t - staff)))

        # ВОРОНКА ПОДБОРА
        src = t - contact_lag_months
        if src >= 0 and hire_arr[t] > 0:
            hire_value = float(hire_arr[t])
            k_sum = float(ksum_col[src])
            k_stage1 = float(k_stage1_col[src])

            if k_sum > 0 and k_stage1 > 0:
                contacts = hire_value / k_sum
                contacts_need[src] += contacts
                candidates = contacts / k_stage1
                candidates_need[src] += candidates

    # Финальная таблица для вывода
    res = pd.DataFrame({
        "date": months,
        "staff_plan": staff_plan.astype(int),
        "staff_forecast": staff_by_month.astype(int),
        "deficit": deficit_arr.astype(int),
        "hire_out": hire_out_arr.astype(int),
        "hire_need": hire_arr.astype(int),
        "contacts_need": np.round(contacts_need, 0).astype(int),
        "candidates_need": np.round(candidates_need, 0).astype(int),})

    if has_internship:
        res["interns_start"] = interns_start_arr.astype(int)
        res["interns_to_staff"] = interns_to_staff_arr.astype(int)
        res["interns_end"] = interns_end_arr.astype(int)

    else:
        res["pending_start"] = pending_start_arr.astype(int)
        res["pending_out"] = pending_out_arr.astype(int)
        res["pending_end"] = pending_end_arr.astype(int)

    # Ограничиваем вывод нужным периодом
    if show_until is not None:
        end_period = pd.Period(show_until, freq="M")
        month_periods = pd.PeriodIndex(pd.to_datetime(res["date"]), freq="M")

        res = res.loc[month_periods <= end_period].reset_index(drop=True)

    return res

In [25]:
# первая проверка для 1 должности со стажерским периодом с января 2026

df_one = forecast_df[forecast_df["vacancy"] == "Инспектор контроля"].copy()

res_one = plan_staff_model(
    df=df_one,
    staff_prev_month=int(df_one["initial_staff"].iloc[0]),
    has_internship=bool(df_one["has_internship"].iloc[0]),

    internship_duration=int(df_one["internship_duration_months"].iloc[0]),
    group_limit=int(df_one["group_limit"].iloc[0]),
    internship_capacity=int(df_one["internship_capacity"].iloc[0]),
    initial_interns=int(df_one["initial_interns"].iloc[0]),

    employees_in_processing=0,
    contact_wait_days=int(df_one["contact_lag_days"].iloc[0]),

    from_month="2026-01",
    show_until="2026-12",)

res_one

,date,staff_plan,staff_forecast,deficit,hire_out,hire_need,contacts_need,candidates_need,interns_start,interns_to_staff,interns_end
0,2026-01,577,546,31,0,45,76,181,28,0,70
1,2026-02,577,536,41,0,8,67,159,70,0,70
2,2026-03,577,525,52,0,7,381,907,70,0,70
3,2026-04,577,545,32,0,40,133,317,70,32,70
4,2026-05,577,537,40,0,14,124,295,70,5,70
5,2026-06,577,527,50,0,13,343,816,70,4,70
6,2026-07,577,540,37,0,36,181,431,70,27,70
7,2026-08,578,536,42,0,19,162,385,70,10,70
8,2026-09,578,533,45,0,17,229,544,70,9,70
9,2026-10,578,546,32,0,24,229,544,70,24,62


In [30]:
# Вывод для бизнес-интерпретации - должность со стажерским периодом: Инспектор контроля

hr_result_one = res_one.copy()
hr_result_one.insert(1, "vacancy", "Инспектор контроля")

hr_result_one = hr_result_one.rename(columns={
    "date": "Месяц",
    "vacancy": "Должность",
    "staff_plan": "Плановая численность основного штата",
    "staff_forecast": "Прогнозная численность основного штата",
    "deficit": "Прогнозный дефицит основного штата",
    "hire_need": "Рекомендуемый набор сотрудников-стажеров",
    "contacts_need": "Необходимое количество контактов",
    "candidates_need": "Необходимое количество соискателей",
    "interns_start": "Сотрудники-стажеры на начало месяца",
    "interns_to_staff": "Переведено из стажеров в основной штат"})

business_columns = [
    "Месяц",
    "Должность",
    "Плановая численность основного штата",
    "Прогнозная численность основного штата",
    "Прогнозный дефицит основного штата",
    "Рекомендуемый набор сотрудников-стажеров",
    "Сотрудники-стажеры на начало месяца",
    "Переведено из стажеров в основной штат",
    "Необходимое количество контактов",
    "Необходимое количество соискателей",]

hr_result_one = hr_result_one[business_columns]
hr_result_one

,Месяц,Должность,Плановая численность основного штата,Прогнозная численность основного штата,Прогнозный дефицит основного штата,Рекомендуемый набор сотрудников-стажеров,Сотрудники-стажеры на начало месяца,Переведено из стажеров в основной штат,Необходимое количество контактов,Необходимое количество соискателей
0,2026-01,Инспектор контроля,577,546,31,45,28,0,76,181
1,2026-02,Инспектор контроля,577,536,41,8,70,0,67,159
2,2026-03,Инспектор контроля,577,525,52,7,70,0,381,907
3,2026-04,Инспектор контроля,577,545,32,40,70,32,133,317
4,2026-05,Инспектор контроля,577,537,40,14,70,5,124,295
5,2026-06,Инспектор контроля,577,527,50,13,70,4,343,816
6,2026-07,Инспектор контроля,577,540,37,36,70,27,181,431
7,2026-08,Инспектор контроля,578,536,42,19,70,10,162,385
8,2026-09,Инспектор контроля,578,533,45,17,70,9,229,544
9,2026-10,Инспектор контроля,578,546,32,24,70,24,229,544


In [27]:
# вторая проверка для 1 должности со стажерским периодом с марта 2026, поэтому значения вводятся вручную

df_two = forecast_df[forecast_df["vacancy"] == "Оператор техники"].copy()
manual_staff_february = 82          # фактический основной штат на конец февраля
manual_accepted_prev_month = 5      # кандидаты, принятые в предыдущем месяце и ожидающие выхода
manual_accepted_current_month = 4   # кандидаты, принятые в текущем месяце и ожидающие выхода


max_hire_value = df_two["max_hire_monthly"].dropna()
max_hire_no_internship = ( int(max_hire_value.iloc[0]) if len(max_hire_value) > 0 else None)

res_two = plan_staff_model(
    df=df_two,
    staff_prev_month=manual_staff_february,
    has_internship=False,

    internship_duration=int(df_two["internship_duration_months"].iloc[0]),
    group_limit=int(df_two["group_limit"].iloc[0]),
    internship_capacity=int(df_two["internship_capacity"].iloc[0]),
    initial_interns=0,

    employees_in_processing=0,
    contact_wait_days=int(df_two["contact_lag_days"].iloc[0]),

    max_hire_no_internship=max_hire_no_internship,
    accepted_prev_month_no_internship=manual_accepted_prev_month,
    accepted_current_month_no_internship=manual_accepted_current_month,

    from_month="2026-03",
    show_until="2026-12",)

res_two

,date,staff_plan,staff_forecast,deficit,hire_out,hire_need,contacts_need,candidates_need,pending_start,pending_out,pending_end
0,2026-03,104,85,19,5,15,47,124,9,5,19
1,2026-04,104,86,18,4,10,47,124,19,4,25
2,2026-05,104,98,6,15,4,47,124,25,15,14
3,2026-06,105,105,0,10,4,35,93,14,10,8
4,2026-07,105,105,0,4,4,35,93,8,4,8
5,2026-08,105,105,0,4,3,35,93,8,4,7
6,2026-09,105,105,0,4,3,0,0,7,4,6
7,2026-10,105,105,0,3,3,0,0,6,3,6
8,2026-11,105,105,0,3,0,0,0,6,3,3
9,2026-12,105,105,0,3,0,0,0,3,3,0


In [32]:
# Вывод для бизнес-интерпретации - должность без стажерского периода: Оператор техники

hr_result_two = res_two.copy()
hr_result_two.insert(1, "vacancy", "Оператор техники")

hr_result_two = hr_result_two.rename(columns={
    "date": "Месяц",
    "vacancy": "Должность",
    "staff_plan": "Плановая численность основного штата",
    "staff_forecast": "Прогнозная численность основного штата",
    "deficit": "Прогнозный дефицит основного штата",
    "hire_out": "Выходы в штат после лага",
    "hire_need": "Рекомендуемый найм",
    "contacts_need": "Необходимое количество контактов",
    "candidates_need": "Необходимое количество соискателей",
    "pending_start": "Кандидаты в ожидании выхода на начало месяца",
    "pending_out": "Кандидаты, вышедшие в штат",})

business_columns_two = [
    "Месяц",
    "Должность",
    "Плановая численность основного штата",
    "Прогнозная численность основного штата",
    "Прогнозный дефицит основного штата",
    "Рекомендуемый найм",
    "Выходы в штат после лага",
    "Кандидаты в ожидании выхода на начало месяца",
    "Кандидаты, вышедшие в штат",
    "Необходимое количество контактов",
    "Необходимое количество соискателей",]

hr_result_two = hr_result_two[business_columns_two]
hr_result_two

,Месяц,Должность,Плановая численность основного штата,Прогнозная численность основного штата,Прогнозный дефицит основного штата,Рекомендуемый найм,Выходы в штат после лага,Кандидаты в ожидании выхода на начало месяца,"Кандидаты, вышедшие в штат",Необходимое количество контактов,Необходимое количество соискателей
0,2026-03,Оператор техники,104,85,19,15,5,9,5,47,124
1,2026-04,Оператор техники,104,86,18,10,4,19,4,47,124
2,2026-05,Оператор техники,104,98,6,4,15,25,15,47,124
3,2026-06,Оператор техники,105,105,0,4,10,14,10,35,93
4,2026-07,Оператор техники,105,105,0,4,4,8,4,35,93
5,2026-08,Оператор техники,105,105,0,3,4,8,4,35,93
6,2026-09,Оператор техники,105,105,0,3,4,7,4,0,0
7,2026-10,Оператор техники,105,105,0,3,3,6,3,0,0
8,2026-11,Оператор техники,105,105,0,0,3,6,3,0,0
9,2026-12,Оператор техники,105,105,0,0,3,3,3,0,0
